## 청년 AI.Big Data 아카데미 예측분석

### Car.csv 종합 실습 최종본

- `Car.csv` raw 데이터 사용
- `종합실습_1(1).ipynb`의 데이터 정제 및 파생변수 그대로 사용
- 연습파일에서 학습한 방법만 사용
- 1단계: 전체 변수 기반 모델링 및 평가
- 2단계: 유의성 / 다중공선성 반영 후 유의미 변수 기반 모델링 및 평가


In [ ]:
import matplotlib

# 맑은 고딕 적용
matplotlib.rc("font", family = "Noto Sans CJK JP")
# 음수 표시
matplotlib.rc("axes", unicode_minus = False)

In [ ]:
# 실행결과 경고메시지 출력 제외
import warnings
warnings.filterwarnings('ignore')

# 데이터 구성 패키지: Series, DataFrame
import pandas as pd
# 행렬 연산 패키지
import numpy as np

# 데이터 시각화 패키지
import matplotlib.pyplot as plt
import seaborn as sns

# 학습용/평가용 데이터 분리
from sklearn.model_selection import train_test_split

# 예측 모델
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# 평가 함수
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 통계 분석
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tools.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

### 데이터 불러오기

In [ ]:
df_raw = pd.read_csv("/home/piai/다운로드/2. Big Data 분석 실습파일/Car.csv")
df_raw.head()

In [ ]:
df_raw.info()

## 1. 종합실습_1(1).ipynb와 동일한 데이터 정제

In [ ]:
# New_Price 열 제거
df_raw = df_raw.drop("New_Price", axis = 1)
df_raw.head()

In [ ]:
df_raw[(df_raw["Mileage"].isnull() == True)]

In [ ]:
display(df_raw[(df_raw["Name"] == "Toyota Prius 2009-2016 Z4")])
df_raw[(df_raw["Name"] == "Mahindra E Verito D4")]

In [ ]:
df_raw[df_raw.Seats < 1]

In [ ]:
# 동일 모델 x -> mileage 결측치 제거
df_raw = df_raw.dropna(subset = ["Mileage"])
df_raw = df_raw.dropna(subset = ["Price"])
# Seat = 0 제거
df_raw = df_raw.drop(index = 3999)

df_raw.info()

In [ ]:
# Power 열의 'null bhp' 결측치로 변경
df_raw["Power"] = df_raw["Power"].replace("null bhp", np.nan)
df_raw["Mileage"] = df_raw["Mileage"].replace("0.0 kmpl", np.nan)

df_raw.info()

In [ ]:
# 같은 모델의 제원값 동일하게 채우기
cols = ["Mileage", "Engine", "Power", "Seats"]

df_raw[cols] = (
    df_raw.groupby("Name")[cols]
    .transform(lambda x: x.ffill().bfill())
)

df_raw.info()

In [ ]:
df_raw[["Mileage", "Mileage_UNIT"]] = df_raw["Mileage"].str.split(expand = True)
df_raw[["Engine", "Engine_UNIT"]] = df_raw["Engine"].str.split(expand = True)
df_raw[["Power", "Power_UNIT"]] = df_raw["Power"].str.split(expand = True)

df_raw["Mileage"] = df_raw["Mileage"].astype("float64")
df_raw["Engine"] = df_raw["Engine"].astype("float64")
df_raw["Power"] = df_raw["Power"].astype("float64")

# 단위를 나타내는 항목 제외
df_raw = df_raw.drop(["Mileage_UNIT", "Engine_UNIT", "Power_UNIT"], axis = 1)

df_raw.describe()

In [ ]:
# 빈 결측치 평균값으로 채우기
df_raw.loc[df_raw["Mileage"].isnull() == True, "Mileage"] = 18.382425
df_raw.loc[df_raw["Engine"].isnull() == True, "Engine"] = 1620.027080
df_raw.loc[df_raw["Power"].isnull() == True, "Power"] = 113.140881
df_raw.loc[df_raw["Seats"].isnull() == True, "Seats"] = 5

df_raw.info()

In [ ]:
df_raw["Name"] = df_raw["Name"].apply(
    lambda x: np.nan if pd.isna(x) else ("Land Rover" if str(x).startswith("Land Rover") else str(x).split()[0])
)
df_raw.head()

### 데이터 인코딩

In [ ]:
owner_map = {
    "First": 1,
    "Second": 2,
    "Third": 3,
    "Fourth & Above": 4
}

df_raw["Owner_Type"] = df_raw["Owner_Type"].replace(owner_map)

In [ ]:
trans_map = {
    "Manual": 0,
    "Automatic": 1
}

df_raw["Transmission"] = df_raw["Transmission"].replace(trans_map)

In [ ]:
region_map = {
    "Ahmedabad": 1,
    "Mumbai": 1,
    "Pune": 1,

    "Bangalore": 2,
    "Chennai": 2,
    "Coimbatore": 2,
    "Hyderabad": 2,
    "Kochi": 2,

    "Delhi": 3,
    "Jaipur": 3,

    "Kolkata": 0
}

df_raw["Region"] = df_raw["Location"].replace(region_map)

In [ ]:
brand_price_mean = df_raw.groupby("Name")["Price"].mean()
df_raw["Brand_Price_Mean"] = df_raw["Name"].map(brand_price_mean)

bins = [0, 7000, 10000, 20000, 50000, np.inf]
labels = ["초저가형", "저가형", "중간", "고가형", "초고가형"]

df_raw["Brand_Tier"] = pd.cut(df_raw["Brand_Price_Mean"], bins = bins, labels = labels)

df_raw = pd.get_dummies(df_raw, columns = ["Fuel_Type", "Brand_Tier", "Region"], drop_first = True)
df_raw.drop(["Brand_Price_Mean"], axis = 1, inplace = True)
df_raw.head()

In [ ]:
# 원-핫 인코딩으로 생성된 bool 형태의 데이터를 int(0, 1)로 변환
bool_cols = df_raw.select_dtypes(include = ["bool"]).columns
df_raw[bool_cols] = df_raw[bool_cols].astype(int)

df_raw.info()

## 2. 종합실습_1(1).ipynb와 동일한 이상치 처리 및 파생변수

In [ ]:
# Kilometers_Driven 이상치 제거
df_raw.boxplot("Kilometers_Driven")
plt.show()

In [ ]:
df_raw[df_raw["Kilometers_Driven"] >= 6000000]

In [ ]:
df_raw = df_raw.drop(index = 2328)

df_raw.boxplot("Kilometers_Driven")
plt.show()

In [ ]:
df_raw.boxplot("Power")
plt.show()

df_raw.boxplot("Mileage")
plt.show()

df_raw.boxplot("Engine")
plt.show()

df_raw.boxplot("Price")
plt.show()

In [ ]:
# 차 나이
df_raw["car_age"] = 2020 - df_raw["Year"]

# 연간주행거리
df_raw["kilometerPerYear"] = df_raw["Kilometers_Driven"] / df_raw["car_age"]
df_raw["kilometerPerYear"] = df_raw["kilometerPerYear"].round(2)

# 차량크기
car_size_map = {
    2: 0, 3: 0, 4: 0,
    5: 1, 6: 1, 7: 1,
    8: 2, 9: 2, 10: 2
}
df_raw["car_size"] = df_raw["Seats"].map(car_size_map)

# 인구수
population_map = {
    "Delhi": 2,
    "Kolkata": 2,
    "Mumbai": 2,
    "Bangalore": 1,
    "Chennai": 1,
    "Hyderabad": 1,
    "Jaipur": 0,
    "Ahmedabad": 0,
    "Pune": 0,
    "Coimbatore": 0,
    "Kochi": 0
}
df_raw["population"] = df_raw["Location"].replace(population_map)

df_raw.head()

In [ ]:
# Price 로그 변환
sns.histplot(data = df_raw, x = "Price")
plt.show()

df_raw["Price"] = np.log(df_raw["Price"])

sns.histplot(data = df_raw, x = "Price")
plt.show()

## 3. 변수 관계 분석

In [ ]:
sns.pairplot(data = df_raw, y_vars = ["Price"], x_vars = ["Year", "Kilometers_Driven", "Transmission"])
plt.show()

sns.pairplot(data = df_raw, y_vars = ["Price"], x_vars = ["Owner_Type", "Mileage", "Engine"])
plt.show()

sns.pairplot(data = df_raw, y_vars = ["Price"], x_vars = ["Power", "Seats"])
plt.show()

sns.pairplot(data = df_raw, y_vars = ["Price"], x_vars = ["car_age", "kilometerPerYear", "car_size", "population"])
plt.show()

In [ ]:
num_cols = ["Price", "car_age", "kilometerPerYear", "Engine", "Power", "Mileage"]

plt.figure(figsize = (10, 8))
sns.heatmap(df_raw[num_cols].corr(),
            annot = True,
            cmap = "coolwarm",
            fmt = ".2f",
            linewidths = 0.5,
            vmin = -1, vmax = 1)

plt.title("주요 수치형 변수 간의 상관관계 히트맵", fontsize = 15)
plt.show()

In [ ]:
brand_cols = ["Brand_Tier_저가형", "Brand_Tier_중간", "Brand_Tier_고가형", "Brand_Tier_초고가형"]

df_brand = df_raw.melt(id_vars = ["Price"], value_vars = brand_cols,
                       var_name = "Brand_Tier", value_name = "Value")
df_brand = df_brand[df_brand["Value"] == 1]

plt.figure(figsize = (10, 6))
sns.boxplot(x = "Brand_Tier", y = "Price", data = df_brand, order = brand_cols, palette = "Set2")
plt.title("브랜드 등급별 중고차 가격 분포", fontsize = 15)
plt.ylabel("Price")
plt.xlabel("브랜드 등급")
plt.show()

## 4. 전체 변수 사용 모델링

In [ ]:
df_raw_x = df_raw.drop(["Price", "Name", "Location"], axis = 1)
df_raw_y = df_raw["Price"]

df_train_x, df_test_x, df_train_y, df_test_y = train_test_split(
    df_raw_x, df_raw_y, test_size = 0.3, random_state = 1234
)

print("train data X size : {}".format(df_train_x.shape))
print("train data Y size : {}".format(df_train_y.shape))
print("test data X size : {}".format(df_test_x.shape))
print("test data Y size : {}".format(df_test_y.shape))

### 규제화회귀분석

In [ ]:
ridge = Ridge(alpha = 1)
ridge.fit(df_train_x, df_train_y)
df_ridge_coef = pd.DataFrame({"Coef": ridge.coef_}, index = df_train_x.columns)
df_ridge_coef

In [ ]:
df_ridge_coef.plot.barh(y = "Coef", legend = False)

In [ ]:
train_pred = ridge.predict(df_train_x)
test_pred = ridge.predict(df_test_x)

print("Ridge train data의 결정계수:", r2_score(df_train_y, train_pred))
print("Ridge test data의 결정계수:", r2_score(df_test_y, test_pred))
print("Ridge test RMSE:", mean_squared_error(df_test_y, test_pred) ** 0.5)

In [ ]:
lasso = Lasso(alpha = 0.01, max_iter = 10000)
lasso.fit(df_train_x, df_train_y)
df_lasso_coef = pd.DataFrame({"Coef": lasso.coef_}, index = df_train_x.columns)
df_lasso_coef

In [ ]:
elastic = ElasticNet(alpha = 0.01, l1_ratio = 0.5, max_iter = 10000)
elastic.fit(df_train_x, df_train_y)
df_elastic_coef = pd.DataFrame({"Coef": elastic.coef_}, index = df_train_x.columns)
df_elastic_coef

In [ ]:
for model_name, model in [("Lasso", lasso), ("ElasticNet", elastic)]:
    train_pred = model.predict(df_train_x)
    test_pred = model.predict(df_test_x)
    print(model_name, "train data의 결정계수:", r2_score(df_train_y, train_pred))
    print(model_name, "test data의 결정계수:", r2_score(df_test_y, test_pred))
    print(model_name, "test RMSE:", mean_squared_error(df_test_y, test_pred) ** 0.5)
    print()

### 의사결정나무

In [ ]:
tree_uncustomized = DecisionTreeRegressor(random_state = 1234)
tree_uncustomized.fit(df_train_x, df_train_y)

print("Train Score : {:.3f}".format(tree_uncustomized.score(df_train_x, df_train_y)))
print("Test Score  : {:.3f}".format(tree_uncustomized.score(df_test_x, df_test_y)))

In [ ]:
train_score = []; test_score = []
para_leaf = [n_leaf * 2 for n_leaf in range(1, 11)]

for v_min_samples_leaf in para_leaf:
    tree = DecisionTreeRegressor(random_state = 1234, min_samples_leaf = v_min_samples_leaf)
    tree.fit(df_train_x, df_train_y)
    train_score.append(tree.score(df_train_x, df_train_y))
    test_score.append(tree.score(df_test_x, df_test_y))

df_score_leaf = pd.DataFrame()
df_score_leaf["MinSamplesLeaf"] = para_leaf
df_score_leaf["TrainScore"] = train_score
df_score_leaf["TestScore"] = test_score
df_score_leaf.round(3)

In [ ]:
plt.plot(para_leaf, train_score, linestyle = "-", label = "Train Score")
plt.plot(para_leaf, test_score, linestyle = "--", label = "Test Score")
plt.xlabel("min samples leaf"); plt.ylabel("score")
plt.legend()

In [ ]:
tree_final = DecisionTreeRegressor(random_state = 1234, min_samples_leaf = 8)
tree_final.fit(df_train_x, df_train_y)

tree_pred = tree_final.predict(df_test_x)
print("DecisionTree test data의 결정계수:", r2_score(df_test_y, tree_pred))
print("DecisionTree test RMSE:", mean_squared_error(df_test_y, tree_pred) ** 0.5)

### 랜덤포레스트

In [ ]:
rf_uncustomized = RandomForestRegressor(random_state = 1234)
rf_uncustomized.fit(df_train_x, df_train_y)

print("Train Score : {:.3f}".format(rf_uncustomized.score(df_train_x, df_train_y)))
print("Test Score  : {:.3f}".format(rf_uncustomized.score(df_test_x, df_test_y)))

In [ ]:
train_score = []; test_score = []
para_n_tree = [n_tree * 10 for n_tree in range(1, 11)]

for v_n_estimators in para_n_tree:
    rf = RandomForestRegressor(n_estimators = v_n_estimators, random_state = 1234)
    rf.fit(df_train_x, df_train_y)
    train_score.append(rf.score(df_train_x, df_train_y))
    test_score.append(rf.score(df_test_x, df_test_y))

df_score_n = pd.DataFrame()
df_score_n["n_estimators"] = para_n_tree
df_score_n["TrainScore"] = train_score
df_score_n["TestScore"] = test_score
df_score_n.round(3)

In [ ]:
plt.plot(para_n_tree, train_score, linestyle = "-", label = "Train Score")
plt.plot(para_n_tree, test_score, linestyle = "--", label = "Test Score")
plt.ylabel("score"); plt.xlabel("n_estimators")
plt.legend()

In [ ]:
train_score = []; test_score = []
para_leaf = [n_leaf * 2 for n_leaf in range(1, 11)]

for v_min_samples_leaf in para_leaf:
    rf = RandomForestRegressor(n_estimators = 100, min_samples_leaf = v_min_samples_leaf, random_state = 1234)
    rf.fit(df_train_x, df_train_y)
    train_score.append(rf.score(df_train_x, df_train_y))
    test_score.append(rf.score(df_test_x, df_test_y))

df_score_leaf = pd.DataFrame()
df_score_leaf["MinSamplesLeaf"] = para_leaf
df_score_leaf["TrainScore"] = train_score
df_score_leaf["TestScore"] = test_score
df_score_leaf.round(3)

In [ ]:
rf_final = RandomForestRegressor(
    n_estimators = 100, max_depth = 10, min_samples_leaf = 4, random_state = 1234
)
rf_final.fit(df_train_x, df_train_y)

rf_pred = rf_final.predict(df_test_x)
print("RandomForest test data의 결정계수:", r2_score(df_test_y, rf_pred))
print("RandomForest test RMSE:", mean_squared_error(df_test_y, rf_pred) ** 0.5)

### 그래디언트부스팅

In [ ]:
gb_uncustomized = GradientBoostingRegressor(random_state = 1234)
gb_uncustomized.fit(df_train_x, df_train_y)

print("Train Score : {:.3f}".format(gb_uncustomized.score(df_train_x, df_train_y)))
print("Test Score  : {:.3f}".format(gb_uncustomized.score(df_test_x, df_test_y)))

In [ ]:
train_score = []; test_score = []
para_n_tree = [n_tree * 10 for n_tree in range(1, 11)]

for v_n_estimators in para_n_tree:
    gb = GradientBoostingRegressor(n_estimators = v_n_estimators, random_state = 1234)
    gb.fit(df_train_x, df_train_y)
    train_score.append(gb.score(df_train_x, df_train_y))
    test_score.append(gb.score(df_test_x, df_test_y))

df_score_n = pd.DataFrame()
df_score_n["n_estimators"] = para_n_tree
df_score_n["TrainScore"] = train_score
df_score_n["TestScore"] = test_score
df_score_n.round(3)

In [ ]:
train_score = []; test_score = []
para_depth = [depth for depth in range(1, 11)]

for v_max_depth in para_depth:
    gb = GradientBoostingRegressor(n_estimators = 100, max_depth = v_max_depth,
                                   learning_rate = 0.05, random_state = 1234)
    gb.fit(df_train_x, df_train_y)
    train_score.append(gb.score(df_train_x, df_train_y))
    test_score.append(gb.score(df_test_x, df_test_y))

df_score_depth = pd.DataFrame()
df_score_depth["Depth"] = para_depth
df_score_depth["TrainScore"] = train_score
df_score_depth["TestScore"] = test_score
df_score_depth.round(3)

In [ ]:
gb_final = GradientBoostingRegressor(
    n_estimators = 100, learning_rate = 0.05, max_depth = 4, random_state = 1234
)
gb_final.fit(df_train_x, df_train_y)

gb_pred = gb_final.predict(df_test_x)
print("GradientBoosting test data의 결정계수:", r2_score(df_test_y, gb_pred))
print("GradientBoosting test RMSE:", mean_squared_error(df_test_y, gb_pred) ** 0.5)

### 전체 변수 모델 비교

In [ ]:
models = ["Ridge", "Lasso", "ElasticNet", "DecisionTree", "RandomForest", "GradientBoosting"]
pred_list = [
    ridge.predict(df_test_x),
    lasso.predict(df_test_x),
    elastic.predict(df_test_x),
    tree_final.predict(df_test_x),
    rf_final.predict(df_test_x),
    gb_final.predict(df_test_x)
]

r2 = []
rmse = []
mae = []

for pred in pred_list:
    r2.append(r2_score(df_test_y, pred))
    rmse.append(mean_squared_error(df_test_y, pred) ** 0.5)
    mae.append(mean_absolute_error(df_test_y, pred))

df_model_all = pd.DataFrame({
    "Model": models,
    "R2": r2,
    "RMSE": rmse,
    "MAE": mae
}).sort_values("R2", ascending = False)

df_model_all.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (13, 4))

ax[0].bar(df_model_all["Model"], df_model_all["R2"])
ax[0].set_title("전체 변수 모델별 R2")
ax[0].tick_params(axis = "x", rotation = 45)

ax[1].bar(df_model_all["Model"], df_model_all["RMSE"])
ax[1].set_title("전체 변수 모델별 RMSE")
ax[1].tick_params(axis = "x", rotation = 45)

plt.tight_layout()
plt.show()

## 5. 유의성과 다중공선성 분석

In [ ]:
df_stat = df_raw.drop(["Name", "Location"], axis = 1)
df_stat_x = df_stat.drop("Price", axis = 1)
df_stat_y = df_stat["Price"]

df_stat_x_const = add_constant(df_stat_x)
reg_model = sm.OLS(df_stat_y, df_stat_x_const)
reg_result = reg_model.fit()
print(reg_result.summary())

In [ ]:
# 다중공선성 확인
vif = pd.DataFrame()
vif["VIF"] = [variance_inflation_factor(df_stat_x.values, i) for i in range(df_stat_x.shape[1])]
vif["Variable"] = df_stat_x.columns
vif.sort_values("VIF", ascending = False)

In [ ]:
# p-value와 VIF 기준으로 유의미 변수 선택
sig_var = reg_result.pvalues.drop("const")
sig_var = sig_var[sig_var < 0.05].index.tolist()

vif_var = vif[vif["VIF"] < 10]["Variable"].tolist()

selected_var = [col for col in df_stat_x.columns if col in sig_var and col in vif_var]

print("유의한 변수 개수:", len(sig_var))
print("VIF 10 미만 변수 개수:", len(vif_var))
print("최종 선택 변수 개수:", len(selected_var))
print(selected_var)

## 6. 유의미 변수만 사용한 모델링

In [ ]:
df_sig_x = df_stat_x[selected_var]
df_sig_y = df_stat_y

df_train_x, df_test_x, df_train_y, df_test_y = train_test_split(
    df_sig_x, df_sig_y, test_size = 0.3, random_state = 1234
)

print("train data X size : {}".format(df_train_x.shape))
print("test data X size : {}".format(df_test_x.shape))

### 규제화회귀분석

In [ ]:
ridge_sig = Ridge(alpha = 1)
lasso_sig = Lasso(alpha = 0.01, max_iter = 10000)
elastic_sig = ElasticNet(alpha = 0.01, l1_ratio = 0.5, max_iter = 10000)

for model in [ridge_sig, lasso_sig, elastic_sig]:
    model.fit(df_train_x, df_train_y)

df_elastic_sig_coef = pd.DataFrame({"Coef": elastic_sig.coef_}, index = df_train_x.columns)
df_elastic_sig_coef["AbsCoef"] = abs(df_elastic_sig_coef["Coef"])
df_elastic_sig_coef.sort_values("AbsCoef", ascending = False)

### 의사결정나무 / 랜덤포레스트 / 그래디언트부스팅

In [ ]:
tree_sig = DecisionTreeRegressor(random_state = 1234, min_samples_leaf = 8)
rf_sig = RandomForestRegressor(n_estimators = 100, max_depth = 10, min_samples_leaf = 4, random_state = 1234)
gb_sig = GradientBoostingRegressor(n_estimators = 100, learning_rate = 0.05, max_depth = 4, random_state = 1234)

tree_sig.fit(df_train_x, df_train_y)
rf_sig.fit(df_train_x, df_train_y)
gb_sig.fit(df_train_x, df_train_y)

In [ ]:
models = ["Ridge", "Lasso", "ElasticNet", "DecisionTree", "RandomForest", "GradientBoosting"]
pred_list = [
    ridge_sig.predict(df_test_x),
    lasso_sig.predict(df_test_x),
    elastic_sig.predict(df_test_x),
    tree_sig.predict(df_test_x),
    rf_sig.predict(df_test_x),
    gb_sig.predict(df_test_x)
]

r2 = []
rmse = []
mae = []

for pred in pred_list:
    r2.append(r2_score(df_test_y, pred))
    rmse.append(mean_squared_error(df_test_y, pred) ** 0.5)
    mae.append(mean_absolute_error(df_test_y, pred))

df_model_sig = pd.DataFrame({
    "Model": models,
    "R2": r2,
    "RMSE": rmse,
    "MAE": mae
}).sort_values("R2", ascending = False)

df_model_sig.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (13, 4))

ax[0].bar(df_model_sig["Model"], df_model_sig["R2"])
ax[0].set_title("유의미 변수 모델별 R2")
ax[0].tick_params(axis = "x", rotation = 45)

ax[1].bar(df_model_sig["Model"], df_model_sig["RMSE"])
ax[1].set_title("유의미 변수 모델별 RMSE")
ax[1].tick_params(axis = "x", rotation = 45)

plt.tight_layout()
plt.show()

## 7. Price 영향 변수 분석

In [ ]:
# ElasticNet 기준 영향 정도
df_elastic_sig_coef.sort_values("AbsCoef", ascending = False).head(15)

In [ ]:
# RandomForest 기준 영향 정도
df_rf_importance = pd.DataFrame({
    "Feature": df_train_x.columns,
    "Importance": rf_sig.feature_importances_
})
df_rf_importance.sort_values("Importance", ascending = False).head(15)

In [ ]:
# GradientBoosting 기준 영향 정도
df_gb_importance = pd.DataFrame({
    "Feature": df_train_x.columns,
    "Importance": gb_sig.feature_importances_
})
df_gb_importance.sort_values("Importance", ascending = False).head(15)

In [ ]:
plt.figure(figsize = (10, 6))
top_rf = df_rf_importance.sort_values("Importance", ascending = True).tail(15)
plt.barh(top_rf["Feature"], top_rf["Importance"])
plt.title("RandomForest 기준 Price 영향 변수")
plt.show()

plt.figure(figsize = (10, 6))
top_gb = df_gb_importance.sort_values("Importance", ascending = True).tail(15)
plt.barh(top_gb["Feature"], top_gb["Importance"])
plt.title("GradientBoosting 기준 Price 영향 변수")
plt.show()

## 8. 해석

1. 전체 변수 사용 시 `RandomForest`와 `GradientBoosting`의 성능이 가장 우수한지 비교표에서 확인한다.
2. OLS summary의 p-value와 VIF 결과를 이용해 유의미 변수만 선택한다.
3. 유의미 변수만 사용했을 때도 트리 기반 모델이 여전히 높게 나오면 비선형 관계가 강하다고 해석할 수 있다.
4. ElasticNet 회귀계수의 절대값, RandomForest 중요도, GradientBoosting 중요도를 함께 보면 `Price`에 영향을 많이 주는 변수를 확인할 수 있다.
5. 상관계수, 회귀계수, 변수 중요도를 종합하여 `Engine`, `Power`, `Kilometers_Driven`, `Transmission`, `Brand_Tier`, `kilometerPerYear`, `population` 등의 영향 방향과 크기를 해석한다.
